In [1]:
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

import sys; sys.path.insert(0, '..')
from src.palette import register, content_colors, CONTENT_ORDER

CONTENT_COLORS = register('light')

In [2]:
conn = sqlite3.connect('../data/lafc_content.db')

In [3]:
with open('../sql/videos_vs_lafc_match_context.sql') as f:
    query = f.read()

df = pd.read_sql(query, conn)
df = df[df['published_at'] >= '2025-01-01T00:00:00Z'].copy()
df

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,engagement_rate,format,...,goals_for,goals_against,lafc_points,lafc_played,lafc_wins,opp_points,opp_played,opp_wins,days_since_match,days_until_match
0,IxrFLgowFd4,Armindo Sieb is Black & Gold.,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T16:40:02Z,PT52S,275,23,7,0.10909,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.72,NaN
1,HugEGKBw0kk,LAFC vs QRO | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T09:02:51Z,PT14M19S,306,22,26,0.15686,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.40,NaN
2,pLVoxNTyGLI,The top scorer in Leagues Cup history 📈,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:30:24Z,PT15S,6287,169,12,0.02879,short,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.33,NaN
3,wz6UrdGQWjY,BOUANGA EQUALIZER 💥,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:15:01Z,PT13S,3940,91,4,0.02411,short,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.32,NaN
4,SqgJkPzCN6Y,Denis Bouanga equalizes against Querétaro,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:02:06Z,PT13S,1712,55,6,0.03563,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.31,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,zPYzE9H0Zyo,Igor Jesus is Black & Gold,📝 #LAFC acquires Igor Jesus from Portuguese Pr...,2025-01-21T22:36:06Z,PT1M24S,1221,43,5,0.03931,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,93.90,31.96
1211,Uycrl281Zjg,"Part of our History | Thank you, Erik Dueñas",The LAFC Original. Forever Black & Gold.\n\nBe...,2025-01-14T19:59:43Z,PT1M30S,1518,43,6,0.03228,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,86.79,39.07
1212,xkQPq_y1QOg,And now for some midfield thunder 🔨⚡,Odin Thiago Holm is Balck & Gold.\n\nWatch LAF...,2025-01-13T19:44:09Z,PT19S,2473,143,12,0.06268,short,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,85.78,40.08
1213,bm7P0aOOhAo,Odin Thiago Holm is Black & Gold,LAFC has acquired Norwegian midfielder Odin Th...,2025-01-13T19:34:11Z,PT1M23S,2684,67,8,0.02794,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,85.77,40.09


In [4]:
df.shape

(1215, 29)

In [5]:
df.columns

Index(['video_id', 'title', 'description', 'published_at', 'duration',
       'view_count', 'like_count', 'comment_count', 'engagement_rate',
       'format', 'playlist', 'n_playlists', 'all_playlists', 'content_type',
       'season', 'kickoff_utc', 'opponent', 'result', 'home_away', 'goals_for',
       'goals_against', 'lafc_points', 'lafc_played', 'lafc_wins',
       'opp_points', 'opp_played', 'opp_wins', 'days_since_match',
       'days_until_match'],
      dtype='str')

In [6]:
fig = px.histogram(
    df, x='view_count',
    title="Videos per view - linear scale"
    )

fig.update_xaxes(title='Views')

fig.update_yaxes(title='Number of videos')

fig.show()

In [7]:
df['log10_views'] = np.log10(df['view_count'])

fig = px.histogram(
    df, x='log10_views',
    nbins=80,
    title='Videos per View - Log Scale', 
    )
fig.update_xaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'],
    title='Views')

fig.update_yaxes(
    title='Number of videos'
)

fig.show()

In [34]:
fig = px.histogram(
    df, x='engagement_rate',
    nbins=80,
    title= 'Engagement rate per video'
    )

fig.update_yaxes()

fig.show()



In [8]:
df['format'].value_counts()

format
horizontal    741
short         398
live           76
Name: count, dtype: int64

In [9]:
df.groupby('format')['view_count'].median().sort_values(ascending=False)

format
short         8331.0
live          2023.0
horizontal    1700.0
Name: view_count, dtype: float64

In [20]:
df['playback_type'] = np.where(df['format'] == 'short', 'short', 'horizontal+live')

order = ['short', 'horizontal+live']
n = df['playback_type'].value_counts()

fig = px.box(
    df, x='playback_type', y='view_count',
    log_y=True,
    color='playback_type',
    category_orders={'playback_type': order},
    title='View Count by Playback Type - Short vs. Horizontal+Live',
    labels={'playback_type': '', 'view_count': 'View Count'},
)

fig.update_traces(marker=dict(opacity=0.3, size=4), jitter=0.4)

fig.update_layout(showlegend=False)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[100, 1_000, 10_000, 100_000, 1_000_000],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [22]:
order = ['short', 'horizontal+live']
n = df['playback_type'].value_counts()

fig = px.box(
    df, x='playback_type', y='engagement_rate',
    color='playback_type',
    category_orders={'playback_type': order},
    title='Engagement Rate by Playback Type - Short vs. Horizontal+Live',
    labels={'playback_type': '', 'engagement_rate': 'Engagement Rate'},
)

fig.update_traces(marker=dict(opacity=0.3, size=4), jitter=0.4)

fig.update_layout(showlegend=False)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [30]:
thing = pd.crosstab(df['format'], df['content_type'])
thing

content_type,community,feature,full_match,highlights,match_preview,podcast,press_interview,show,unclassified
format,,,,,,,,,
horizontal,1,5,6,171,30,184,155,82,50
live,0,0,2,0,0,69,0,0,2
short,0,0,0,46,1,0,0,0,29


In [32]:
order = df.groupby('content_type')['view_count'].median().sort_values().index.tolist()
n = df['content_type'].value_counts()

fig = px.box(
    df, x='content_type', y='log10_views',
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    category_orders={'content_type': order},
    title='Views (log 10) by Content Type',
    labels={'format_family': 'Format Family', 'log10_views': "Views"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [43]:
unclassified_df = df[df['content_type'] == 'unclassified'].copy()
unclassified_df[['title', 'format', 'playback_type', 'content_type', 'playlist']].sample(30)
unclassified_df['playlist'].value_counts()

playlist
The Son Spotlight    69
Major News            8
The Vela Vault        4
Name: count, dtype: int64

In [ ]:
df.groupby('format_family')['engagement_rate'].median().sort_values()

In [ ]:
order = df.groupby('format_family')['engagement_rate'].median().sort_values().index.tolist()
n = df['format_family'].value_counts()

fig = px.box(
    df, x='format_family', y='engagement_rate',
    color='format_family',
    color_discrete_map=FAMILY_COLORS,
    category_orders={'format_family': order},
    title='Engagement Rate by Format Family',
    labels={'format_family': 'Format Family', 'engagement_rate': 'Engagement Rate'},
    )

for tr in fig.data:
    tr.legendrank = FIXED.index(tr.name)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [ ]:
fig = px.scatter(
    df, x='dur_min', y='view_count',
    log_y=True,
    opacity=0.3,
    trendline='ols',
    hover_data='title',
    title='View Count (log) by Duration in Minutes',
    labels={'view_count': 'View count', 'dur_min': 'Duration in Minutes'}
    )
fig.show()

print(f'R is {df['dur_min'].corr(df['view_count'])}')


In [ ]:
df['log10_dur']   = np.log10(df['dur_min'])

In [ ]:
fig = px.scatter(
    df, x='dur_min', y='view_count',
    log_y=True,
    log_x=True,
    opacity=0.3,
    trendline='ols',
    trendline_options=dict(log_x=True, log_y=True),   # ← fit in log space
    hover_data='title',
    title='View Count (log) by Duration in Minutes (log)',
    labels={'view_count': 'View count', 'dur_min': 'Duration in Minutes (log)'}
    )

fig.update_xaxes(
    tickvals=[0.25, 0.5, 1, 2, 5, 10, 30, 60, 120],
    ticktext=['15s', '30s', '1m', '2m', '5m', '10m', '30m', '1h', '2h'],
    title='Duration')

fig.show()

print(f"R is {df['log10_dur'].corr(df['log10_views']):.3f}")

In [ ]:
fig = px.scatter(
    df, x='log10_dur', y='engagement_rate',
    opacity=0.3,
    hover_data='title',
    trendline='lowess',
    trendline_options=dict(frac=0.3),
    title='Engagement Rate by Duration in Minutes',
    labels={'engagement_rate': 'Engagement Rate', 'dur_min': "Duration in Minutes"}
)

fig.update_xaxes(
    tickvals=[np.log10(v) for v in [0.25, 1, 5, 30, 120]],
    ticktext=['15s', '1m', '5m', '30m', '2h'],
    title='Duration')
fig.update_yaxes(tickformat='.1%')

fig.show()

In [ ]:
# Assigns the columns into two series.
after  = df['days_since_match']    # days SINCE the previous match (always ≥ 0)
before = df['days_until_match']    # days UNTIL the next match     (always ≥ 0)


# Fills in na with infinity, then compares the two series, row by row, and returns a boolean. If before is smaller - then it returns true, meaning, this row is closer to the NEXT match.
closer_to_next = before.fillna(np.inf) < after.fillna(np.inf)

# Assigns either a -before or after based on the boolean in closer_to_next
df['days_from_match'] = np.where(closer_to_next, -before, after)

#Overwrite nan if its been 21 days since (and until) the closest matches

OFFSEASON_DAYS = 21
not_in_cycle = ((after.fillna(np.inf)  > OFFSEASON_DAYS) &
                (before.fillna(np.inf) > OFFSEASON_DAYS))

df.loc[not_in_cycle, 'days_from_match'] = np.nan

print('not in a cycle :', not_in_cycle.sum())
print('before a match :', (df['days_from_match'] < 0).sum())
print('after a match  :', (df['days_from_match'] > 0).sum())

In [ ]:
#Binning and adding the bin info back to the df.

CYCLE_EDGES  = [-np.inf, -7, -3, -1, 0, 2, 4, 8, np.inf]
CYCLE_LABELS = ['7+ before', '3-7 before', '1-3 before', 'matchday',
                '0-1 after', '2-3 after', '4-7 after', '8+ after']

df['cycle_bin'] = pd.cut(
    df['days_from_match'],
    bins=CYCLE_EDGES,
    labels=CYCLE_LABELS, right=False
    )

display(df[['title', 'days_since_match', 'days_until_match', 'cycle_bin']])

In [ ]:
print(df['cycle_bin'].value_counts(dropna=False).sort_index())

In [ ]:
PLOT_BINS = CYCLE_LABELS[1:-1]      # drop '7+ before' and '8+ after'

plot_df = df[df['cycle_bin'].isin(PLOT_BINS)].dropna(subset=['engagement_rate']).copy()
plot_df['cycle_bin'] = plot_df['cycle_bin'].cat.remove_unused_categories()
n = plot_df['cycle_bin'].value_counts()

In [ ]:
fig = px.box(
    plot_df, x='cycle_bin', y='engagement_rate',
    category_orders={'cycle_bin': PLOT_BINS},
    color_discrete_sequence=['#2a78d6'],
    points=False,
    title='Engagement Rate Across the Match Cycle',
    labels={'cycle_bin': 'Position in match cycle',
            'engagement_rate': 'Engagement Rate'},
)

fig.update_xaxes(
    tickvals=PLOT_BINS,
    ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS])

fig.update_yaxes(tickformat='.1%')

fig.add_vline(x=2.5, line_dash='dot', line_color='#888888',
              annotation_text='kickoff', annotation_position='top')

fig.show()

In [ ]:
plot_df.groupby('cycle_bin')['engagement_rate'].median()

In [ ]:
fig = px.box(
    plot_df, x='cycle_bin', y='view_count',
    log_y=True,
    category_orders={'cycle_bin': PLOT_BINS},
    color_discrete_sequence=['#2a78d6'],
    points=False,
    title='View Count Across the Match Cycle',
    labels={'cycle_bin': 'Position in match cycle',
            'view_count': 'View Count (log)'},
)

fig.update_xaxes(
    tickvals=PLOT_BINS,
    ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS])

fig.add_vline(x=2.5, line_dash='dot', line_color='#888888',
              annotation_text='kickoff', annotation_position='top')

fig.show()

In [ ]:
p = df[df['cycle_bin'].isin(PLOT_BINS)].dropna(
        subset=['engagement_rate', 'log10_views']).copy()
p['cycle_bin'] = p['cycle_bin'].cat.remove_unused_categories()
n = p['cycle_bin'].value_counts()

# Two stacked panels sharing one x-axis: same cycle positions, two measures.
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('Engagement rate', 'Views (log₁₀)'))

fig.add_trace(go.Box(x=p['cycle_bin'], y=p['engagement_rate'],
                     marker_color='#2a78d6', boxpoints=False,
                     name='Engagement rate'), row=1, col=1)

fig.add_trace(go.Box(x=p['cycle_bin'], y=p['log10_views'],
                     marker_color='#eb6834', boxpoints=False,
                     name='Views'), row=2, col=1)

for r in (1, 2):
    fig.update_xaxes(categoryorder='array', categoryarray=PLOT_BINS, row=r, col=1)

# Sample sizes only on the bottom panel, since the x-axis is shared.
fig.update_xaxes(tickvals=PLOT_BINS,
                 ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS], row=2, col=1)

fig.update_yaxes(tickformat='.1%', row=1, col=1)
fig.update_yaxes(tickvals=[2, 3, 4, 5, 6],
                 ticktext=['100', '1K', '10K', '100K', '1M'], row=2, col=1)

fig.add_vline(x=2.5, line_dash='dot', line_color='#888888', row=1, col=1)
fig.add_vline(x=2.5, line_dash='dot', line_color='#888888', row=2, col=1)

fig.update_layout(showlegend=False, height=700,
                  title='Engagement and Views Across the Match Cycle')
fig.show()

In [ ]:
# Year is enough granularity — quarterly medians get noisy for the small families.
df['year'] = pd.to_datetime(df['published_at']).dt.year

d = df[df['year'] >= 2018]        # 2015–17 is only 66 videos, pre-first-season

# Median views per family per year, suppressing cells too thin to trust.
MIN_N = 5
g = (d.groupby(['year', 'format_family'], observed=True)
       .agg(median_log=('log10_views', 'median'), n=('video_id', 'size'))
       .reset_index())
g = g[g['n'] >= MIN_N]

# Channel-wide median, drawn into every panel as a shared baseline.
ref = d.groupby('year')['log10_views'].median().reset_index()

fig = px.line(
    g, x='year', y='median_log',
    facet_col='format_family', facet_col_wrap=4,
    color='format_family', color_discrete_map=FAMILY_COLORS,
    category_orders={'format_family': FIXED},
    markers=True, hover_data='n', height=620,
    title='Median Views by Year and Format Family (dotted = channel median)',
    labels={'median_log': 'Median views', 'year': ''},
)

# Bind a reference line to each family's own axes — facet_col_wrap does NOT
# number rows top-to-bottom, so row/col arithmetic here is a trap.
for tr in list(fig.data):
    fig.add_trace(go.Scatter(
        x=ref['year'], y=ref['log10_views'], mode='lines',
        line=dict(color='#9aa0a6', width=1, dash='dot'),
        showlegend=False, hoverinfo='skip',
        xaxis=tr.xaxis, yaxis=tr.yaxis))

fig.update_yaxes(tickvals=[2, 3, 4, 5], ticktext=['100', '1K', '10K', '100K'])
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(showlegend=False)   # panel titles carry identity
fig.show()